In [9]:
!pip install moshi
%pip install --upgrade transformers

  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 46.0 MB/s  0:00:00eta 0:00:01
Using cached huggingface_hub-2.4.0-py3-none-any.whl (857 kB)
Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (516 kB)
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [huggingface-hub]
    Found existing installation: transformers 5.22.1━━━━━━━━━━ 1/3 [huggingface-hub]
    Uninstalling transformers-5.22.1:m╸━━

In [11]:
# from dataclasses import dataclass

from transformers import KuyutaiSpeechToTextProcessor, KyutaiSpeechToTextForConditionalGeneration
#import time
#import sentencepiece
#import sphn
#import textwrap
#import torch

#from moshi.models import loaders, MimiModel, LMModel, LMGen

import IPython
from datasets import load_dataset, Audio

ImportError: cannot import name 'ResolvedRevision' from 'huggingface_hub' (/usr/local/lib/python3.13/dist-packages/huggingface_hub/__init__.py)

In [ ]:
# load a demonstration datasets
librispeech_dummy = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

In [ ]:
# load the model + feature extractor (for pre-processing the audio)
model = KuyutaiSpeechToTextForProcessor.from_pretrained("kyutai/stt-1b-en_fr-trfs")
feature_extractor = AutoFeatureExtractor.from_pretrained("kyutai/stt-1b-en_fr-trfs")

In [ ]:
device = "cuda"
# Use the en+fr low latency model, an alternative is kyutai/stt-2.6b-en
checkpoint_info = loaders.CheckpointInfo.from_hf_repo("kyutai/stt-1b-en_fr")
mimi = checkpoint_info.get_mimi(device=device)
text_tokenizer = checkpoint_info.get_text_tokenizer()
lm = checkpoint_info.get_moshi(device=device)
in_pcms, _ = sphn.read("sample_fr_hibiki_crepes.mp3", sample_rate=mimi.sample_rate)
in_pcms = torch.from_numpy(in_pcms).to(device=device)

stt_config = checkpoint_info.stt_config
pad_left = int(stt_config.get("audio_silence_prefix_seconds", 0.0) * 24000)
pad_right = int((stt_config.get("audio_delay_seconds", 0.0) + 1.0) * 24000)
in_pcms = torch.nn.functional.pad(in_pcms, (pad_left, pad_right), mode="constant")
in_pcms = in_pcms[None, 0:1].expand(1, -1, -1)

state = InferenceState(mimi, text_tokenizer, lm, batch_size=1, device=device)
text = state.run(in_pcms)
print(textwrap.fill(text, width=100))

In [ ]:
from IPython.display import Audio

Audio("sample_fr_hibiki_crepes.mp3")